# Predicción de uso de rutas por tramo (1 a 7 días)

Cuaderno de experimentación con el CSV de prueba de abril. **La versión que corre en Artisa** es el tipo `rutas` de `machine_learning/prediccion.py` (el mismo módulo que el mapa de calor): lee MariaDB, se reentrena cada noche y se ve en la web en Predicción → Rutas.

Diferencias encontradas al auditar este cuaderno (y corregidas en Artisa):
- El objetivo ratio se quedó en **1 árbol** y el modelo **no superó** al promedio del tramo (MAE 0,59 contra 0,57).
- El tramo era el par de celdas de dos reportes seguidos: la misma calle quedaba repartida en muchos tramos según dónde cayó cada reporte. En Artisa se trazan las celdas que cruza el segmento.
- `dia_idx` queda fuera del rango entrenado al predecir; los días se agrupan en UTC y no en hora de Chile.
- El mapa "predicho" mostraba los días de prueba, no días futuros.


## 1. Definir el problema y el objetivo (y)

Instala las librerías base (tablas y gráficos) que se usan en todo el notebook.

In [ ]:
%pip install pandas matplotlib lightgbm folium

## 2. Explorar los datos (EDA)

Carga el CSV de GPS y hace una primera inspección (`info`/`describe`).

In [ ]:
import pandas as pd

df = pd.read_csv("datos_gps_3_semanas.csv"  # CSV de prueba (abril), no se sube a Git)
df.head()

In [ ]:
df.info()
df.describe()

## 3. Limpiar: duplicados, tipos, valores imposibles

eliminar duplicados

In [ ]:
cols_comparar = [
    "report_date",              # fecha y hora del reporte
    "asset_id", "asset_name", "event_name",
    "latitude", "longitude", "location",
    "speed", "cartography_limit_speed",
    "driver_name", "driver_last_name", "driver_document_number",
]

df_original = df.copy()   # respaldo para poder comparar

antes = len(df)
df = df.drop_duplicates(subset=cols_comparar, keep="first").reset_index(drop=True)
print(f"Filas: {antes:,} -> {len(df):,} (eliminadas: {antes - len(df):,})")

Porcentaje que se conservó
Reportes por vehículo, antes de limpiar
Lo mismo, después de limpiar

In [ ]:
print(f"Únicos: {len(df):,} de {len(df_original):,} ({len(df)/len(df_original):.1%})")
print(df_original.groupby("asset_id", observed=True).size().describe())
print(df.groupby("asset_id", observed=True).size().describe())

Cuenta cuántos reportes tiene cada vehículo y muestra los que tienen menos y los que tienen más.

In [ ]:
por_asset = df.groupby(["asset_id", "asset_name"], observed=True).size().sort_values()
print(por_asset.head(10))   # los de menos reportes
print(por_asset.tail(5))    # 

Revisa durante qué fechas reportaron los vehículos con pocos reportes.

In [ ]:
pocos = ["RYCV75", "RYCV76", "TFWK20", "RWSC78", "SWDL61"]
resumen = (df[df["asset_id"].isin(pocos)]
           .groupby("asset_id", observed=True)["report_date"]
           .agg(primero="min", ultimo="max", reportes="count"))
print(resumen)

print("Rango total:", df["report_date"].min(), "->", df["report_date"].max())

Convierte cada columna al tipo de dato que le corresponde, y ordena el DataFrame.


In [ ]:
# --- 3.2 Tipos ---
df["report_date"] = pd.to_datetime(df["report_date"])
df["speed"] = df["speed"].astype("int16")
df["cartography_limit_speed"] = df["cartography_limit_speed"].astype("int16")
for c in ["asset_id", "asset_name", "event_name"]:
    df[c] = df[c].astype("category")
df = df.sort_values(["asset_id", "report_date"]).reset_index(drop=True)
df.info()

eliminamos la columna de fecha de ingesta

In [ ]:
df = df.drop(columns=["ingestado_en"])
df.info()

Falta la parte 3.3. Ejecuta el diagnóstico de valores imposibles 

In [ ]:
print(df[["speed", "cartography_limit_speed", "latitude", "longitude"]].describe())
print("speed < 0:", (df["speed"] < 0).sum())
print("speed > 200:", (df["speed"] > 200).sum())
print("speed > límite:", (df["speed"] > df["cartography_limit_speed"]).sum())
print("lat fuera de Chile:", (~df["latitude"].between(-56, -17)).sum())
print("lon fuera de Chile:", (~df["longitude"].between(-76, -66)).sum())

Los números cuadran: 62.218 − 14.674 = 47.544. Es decir, la mayor parte (76 %) de los “excesos” no eran excesos, eran vehículos en movimiento en tramos sin límite cargado en la cartografía.

In [ ]:
sin_limite = df["cartography_limit_speed"] == 0
print("limite = 0:", sin_limite.sum(), f"({sin_limite.mean():.1%})")

exceso_real = (df["speed"] > df["cartography_limit_speed"]) & ~sin_limite
print("exceso con límite válido:", exceso_real.sum())

# ¿Los excesos reales son moderados o extremos?
exceso = (df["speed"] - df["cartography_limit_speed"])[exceso_real]
print(exceso.describe())

# ¿Velocidad > 0 sin límite? (se mueve sin cartografía)
print("se mueve sin límite:", (sin_limite & (df["speed"] > 0)).sum())

Qué hacer:

Excesos reales: conservarlos. No son errores, y pueden ser lo que quieres modelar.
Límite 0: convertirlo a faltante (NaN) para que el modelo no lo lea como “el límite es 0 km/h”. Los dejo como están, sin borrar filas.
speed: no se toca; no hay negativos ni valores imposibles.

In [ ]:
# Límite 0 = desconocido -> NaN (Int16 con mayúscula admite nulos)
df["cartography_limit_speed"] = (
    df["cartography_limit_speed"].replace(0, pd.NA).astype("Int16")
)

print("Límite faltante:", df["cartography_limit_speed"].isna().sum())

# Revisar los excesos más extremos antes de darlos por buenos
df["exceso"] = df["speed"] - df["cartography_limit_speed"]
print(df.loc[df["exceso"] > 50,
             ["asset_id", "report_date", "speed", "cartography_limit_speed", "location"]].head(10))

Resumen de la limpieza: filas antes/después, tipos de datos cambiados, memoria ahorrada.

In [ ]:
# 4) Reporte: qué se hizo y resultado antes vs después
filas_antes, filas_despues = len(df_original), len(df)
dup_antes = df_original.drop(columns="report_id").duplicated().sum()
dup_despues = df.drop(columns="report_id").duplicated().sum()

print("=== REPORTE DE LIMPIEZA ===\n")
print("1) Duplicados eliminados:", "SÍ" if dup_antes > 0 and dup_despues == 0 else "NO")
print(f"   Filas: {filas_antes:,} -> {filas_despues:,}  ({filas_antes - filas_despues:,} eliminadas)")
print(f"   Duplicados antes: {dup_antes:,} | después: {dup_despues:,}\n")

print("2) Tipos de datos cambiados:")
tipos = pd.DataFrame({"antes": df_original.dtypes.astype(str), "después": df.dtypes.astype(str)})
tipos["¿cambió?"] = tipos["antes"] != tipos["después"]
print(tipos)

mem_antes = df_original.memory_usage(deep=True).sum() / 1e6
mem_despues = df.memory_usage(deep=True).sum() / 1e6
print(f"\n3) Memoria: {mem_antes:.1f} MB -> {mem_despues:.1f} MB")
df.head()

## 4. Predicción de uso de rutas por día (train / validación / test, con early stopping)

Para cada **tramo de ruta** (un segmento corto entre reportes GPS consecutivos, agrupado con los tramos cercanos de otros camiones) y cada día, se predice cuántos camiones *distintos* van a pasar por ahí. Mismo rigor que en la limpieza: **entrenamiento, validación, test, chequeo de overfitting/underfitting con early stopping**, y al final **dos mapas interactivos para comparar** — uno con lo que realmente pasó en los días de test (nunca vistos por el modelo) y otro con lo que el modelo predijo para esos mismos días — para ver si se parecen.

In [ ]:
# ---- 1) Tramos de ruta: segmentos entre reportes consecutivos de cada camion, por dia ----
import numpy as np
import pandas as pd

GRID = 0.01                # tamaño de celda para agrupar tramos cercanos (~1.1 km)
MAX_GAP_MIN = 5             # tramos con mas de 5 min entre reportes no se usan (perdida de señal)
MAX_VEL_IMPLICITA = 150     # km/h; tramos mas rapidos que esto son saltos de GPS, no recorrido real


def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia en linea recta (km) entre dos coordenadas."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


mov = df[df["speed"] > 2].copy()
mov["dia"] = mov["report_date"].dt.date
mov = mov.sort_values(["asset_id", "report_date"])

mov["lat_sig"] = mov.groupby("asset_id", observed=True)["latitude"].shift(-1)
mov["lon_sig"] = mov.groupby("asset_id", observed=True)["longitude"].shift(-1)
mov["t_sig"] = mov.groupby("asset_id", observed=True)["report_date"].shift(-1)

segmentos = mov.dropna(subset=["lat_sig", "lon_sig", "t_sig"]).copy()
segmentos["gap_min"] = (segmentos["t_sig"] - segmentos["report_date"]).dt.total_seconds() / 60
segmentos["dist_km"] = haversine_km(segmentos["latitude"], segmentos["longitude"],
                                     segmentos["lat_sig"], segmentos["lon_sig"])
segmentos["vel_implicita"] = segmentos["dist_km"] / (segmentos["gap_min"] / 60)

validos = segmentos[(segmentos["gap_min"] > 0) &
                     (segmentos["gap_min"] <= MAX_GAP_MIN) &
                     (segmentos["vel_implicita"] <= MAX_VEL_IMPLICITA)].copy()

print(f"Tramos usados: {len(validos):,} / descartados: {len(segmentos) - len(validos):,} "
      f"(mas de {MAX_GAP_MIN} min sin reportes o salto de GPS)")

# Agrupar tramos que caen en la misma celda de grilla (sin importar la direccion): asi el mismo
# pedazo de ruta, recorrido por distintos camiones o en distintos sentidos, es un solo "tramo"
validos["c1_lat"] = (validos["latitude"] / GRID).round().astype(int)
validos["c1_lon"] = (validos["longitude"] / GRID).round().astype(int)
validos["c2_lat"] = (validos["lat_sig"] / GRID).round().astype(int)
validos["c2_lon"] = (validos["lon_sig"] / GRID).round().astype(int)

invertir = (validos["c1_lat"] > validos["c2_lat"]) | \
           ((validos["c1_lat"] == validos["c2_lat"]) & (validos["c1_lon"] > validos["c2_lon"]))
validos["clave"] = np.where(
    invertir,
    validos["c2_lat"].astype(str) + "," + validos["c2_lon"].astype(str) + "_" +
    validos["c1_lat"].astype(str) + "," + validos["c1_lon"].astype(str),
    validos["c1_lat"].astype(str) + "," + validos["c1_lon"].astype(str) + "_" +
    validos["c2_lat"].astype(str) + "," + validos["c2_lon"].astype(str),
)

# Coordenadas representativas de cada tramo (fijas, promedio de todo el historico; para dibujar el mapa)
coords_tramo = validos.groupby("clave").agg(
    lat1=("latitude", "mean"), lon1=("longitude", "mean"),
    lat2=("lat_sig", "mean"), lon2=("lon_sig", "mean"),
)

dias = sorted(validos["dia"].unique())
dia_a_idx = {d: i for i, d in enumerate(dias)}
validos["dia_idx"] = validos["dia"].map(dia_a_idx)

print(f"Dias disponibles: {len(dias)} ({dias[0]} a {dias[-1]})")
print(f"Tramos activos (con historial real): {len(coords_tramo):,}")

In [ ]:
# ---- 2) Armar la tabla (tramo, dia) -> cantidad de camiones distintos, con features sin fuga de datos ----
conteo_dia = (validos.groupby(["clave", "dia_idx"])["asset_id"]
              .nunique().rename("conteo").reset_index())

# Un tramo que un dia no tuvo trafico tambien cuenta (conteo 0): se arma el cruce completo
# tramo activo x dia disponible
tramos_activos = coords_tramo.index.tolist()
combo = pd.MultiIndex.from_product(
    [tramos_activos, range(len(dias))], names=["clave", "dia_idx"]
).to_frame(index=False)

tabla = combo.merge(conteo_dia, on=["clave", "dia_idx"], how="left")
tabla["conteo"] = tabla["conteo"].fillna(0)
tabla = tabla.merge(coords_tramo, on="clave", how="left")
tabla["dia"] = tabla["dia_idx"].map(lambda i: dias[i])
tabla["dia_semana"] = tabla["dia"].map(lambda d: pd.Timestamp(d).weekday())
tabla = tabla.sort_values(["clave", "dia_idx"]).reset_index(drop=True)

# lag1, lag2, tendencia y media historica: SOLO miran dias anteriores al que se esta
# prediciendo (sin fuga). La tendencia (lag1 - lag2) le dice al modelo si el trafico de
# ese tramo viene subiendo o bajando, no solo su nivel.
g_tramo = tabla.groupby("clave")["conteo"]
tabla["lag1"] = g_tramo.shift(1)
tabla["lag2"] = g_tramo.shift(2)
tabla["tendencia"] = tabla["lag1"] - tabla["lag2"]
tabla["media_historica"] = g_tramo.transform(lambda s: s.shift(1).expanding().mean())

# Los primeros 2 dias de cada tramo no tienen historia suficiente: se descartan
tabla = tabla.dropna(subset=["lag1", "lag2", "tendencia", "media_historica"]).reset_index(drop=True)

# Objetivo relativo (no el conteo directo): mas facil de aprender con pocos dias de historia
tabla["objetivo_ratio"] = (tabla["conteo"] + 1) / (tabla["media_historica"] + 1)

print(f"Filas (tramo x dia) para entrenar: {len(tabla):,}")

**Separación en train / validación / test**, por orden de tiempo (nunca al azar — es una serie de tiempo): train para entrenar, validación para el early stopping (elegir cuándo parar de entrenar), y test para la evaluación final, que el modelo no toca para nada durante el entrenamiento.

In [ ]:
feats = ["dia_semana", "dia_idx", "lag1", "lag2", "tendencia", "media_historica"]

dias_idx_disponibles = sorted(tabla["dia_idx"].unique())
n = len(dias_idx_disponibles)
corte_train = dias_idx_disponibles[int(n * 0.60)]
corte_val = dias_idx_disponibles[int(n * 0.80)]

train = tabla[tabla["dia_idx"] <= corte_train]
val = tabla[(tabla["dia_idx"] > corte_train) & (tabla["dia_idx"] <= corte_val)]
test = tabla[tabla["dia_idx"] > corte_val]

print(f"Dias en train: hasta indice {corte_train} ({train['dia'].nunique()} dias, {len(train):,} filas)")
print(f"Dias en validacion: hasta indice {corte_val} ({val['dia'].nunique()} dias, {len(val):,} filas)")
print(f"Dias en test: resto ({test['dia'].nunique()} dias, {len(test):,} filas)")

### Entrenamiento con early stopping

Se entrena mirando el error en **validación** en cada ronda, y se detiene solo cuando deja de mejorar — así no se entrena ni de más (sobreajuste) ni de menos (subajuste).

In [ ]:
import lightgbm as lgb

# min_data_in_leaf mas bajo que antes (30 -> 10): con 16k tramos y pocos dias cada uno,
# un umbral alto le impedia al modelo diferenciar entre tramos.
params = {"objective": "regression", "metric": "l2", "learning_rate": 0.05,
          "num_leaves": 15, "min_data_in_leaf": 10, "seed": 42, "verbose": -1}

# Se prueban dos objetivos y se compara en VALIDACION (no en test) cual predice mejor:
# A) ratio relativo al promedio historico del tramo (lo que ya se usaba)
# B) el conteo directo, sin pasar por el ratio (mas simple, puede ser mejor para tramos
#    con poco trafico historico, donde el ratio se vuelve ruidoso)
ds_train_ratio = lgb.Dataset(train[feats], label=train["objetivo_ratio"])
ds_val_ratio = lgb.Dataset(val[feats], label=val["objetivo_ratio"], reference=ds_train_ratio)
modelo_ratio = lgb.train(params, ds_train_ratio, num_boost_round=2000, valid_sets=[ds_val_ratio],
                          callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])

ds_train_conteo = lgb.Dataset(train[feats], label=train["conteo"])
ds_val_conteo = lgb.Dataset(val[feats], label=val["conteo"], reference=ds_train_conteo)
modelo_conteo = lgb.train(params, ds_train_conteo, num_boost_round=2000, valid_sets=[ds_val_conteo],
                           callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])


def _mae_ratio(conjunto):
    pred = modelo_ratio.predict(conjunto[feats]) * (conjunto["media_historica"] + 1) - 1
    return float(np.abs(np.clip(pred, 0, None) - conjunto["conteo"]).mean())


def _mae_conteo(conjunto):
    pred = np.clip(modelo_conteo.predict(conjunto[feats]), 0, None)
    return float(np.abs(pred - conjunto["conteo"]).mean())


mae_val_ratio = _mae_ratio(val)
mae_val_conteo = _mae_conteo(val)
print(f"Objetivo RATIO  -> rondas optimas: {modelo_ratio.best_iteration:4d}  | MAE validacion: {mae_val_ratio:.3f}")
print(f"Objetivo CONTEO -> rondas optimas: {modelo_conteo.best_iteration:4d}  | MAE validacion: {mae_val_conteo:.3f}")

if mae_val_ratio <= mae_val_conteo:
    print("-> Gana el objetivo RATIO: se usa para el resto del notebook.")
    modelo_tramo = modelo_ratio
    USA_RATIO = True
else:
    print("-> Gana el objetivo CONTEO directo: se usa para el resto del notebook.")
    modelo_tramo = modelo_conteo
    USA_RATIO = False

### Diagnóstico de overfitting / underfitting

Comparamos el error en los tres conjuntos. Si train es mucho mejor que test → sobreajuste. Si los tres son malos → subajuste. Si train/val/test son parecidos y buenos → el modelo generaliza bien.

In [ ]:
def predecir_conteo(conjunto):
    """Predice el conteo esperado, usando el objetivo ganador (ratio o conteo directo)."""
    if USA_RATIO:
        pred = modelo_tramo.predict(conjunto[feats]) * (conjunto["media_historica"] + 1) - 1
    else:
        pred = modelo_tramo.predict(conjunto[feats])
    return np.clip(pred, 0, None)


def error_mae(conjunto):
    pred = predecir_conteo(conjunto)
    return float(np.abs(pred - conjunto["conteo"]).mean())


mae_train = error_mae(train)
mae_val = error_mae(val)
mae_test = error_mae(test)

print("Error absoluto medio (MAE), en camiones distintos por tramo-dia:")
print(f"  Train:      {mae_train:.2f}")
print(f"  Validacion: {mae_val:.2f}")
print(f"  Test:       {mae_test:.2f}")

brecha = mae_test - mae_train
baseline_ingenuo = float(np.abs(test["media_historica"] - test["conteo"]).mean())
print(f"\nReferencia ingenua (predecir siempre el promedio historico de ese tramo): {baseline_ingenuo:.2f}")

if brecha > mae_train:
    print("-> Brecha grande entre train y test: posible SOBREAJUSTE.")
elif mae_test >= baseline_ingenuo:
    print("-> El modelo no supera a la referencia ingenua: posible SUBAJUSTE.")
else:
    print("-> Train, validacion y test son parecidos, y el modelo supera a la referencia: buen ajuste.")

### Comparación contra datos reales (días de test, nunca vistos)

In [ ]:
resultados_test = []
for dia in sorted(test["dia"].unique()):
    sub = test[test["dia"] == dia]
    pred = predecir_conteo(sub)
    real = sub["conteo"].to_numpy()
    corr = float(np.corrcoef(pred, real)[0, 1]) if len(pred) > 1 else float("nan")
    mae_dia = float(np.abs(pred - real).mean())
    resultados_test.append({"dia": dia, "mae": round(mae_dia, 2), "correlacion": round(corr, 3)})

resultados_test_df = pd.DataFrame(resultados_test)
print(resultados_test_df)
print(f"\nCorrelacion promedio prediccion vs realidad (dias de test, nunca vistos): "
      f"{resultados_test_df['correlacion'].mean():.3f}")

### Persistencia del modelo

Guarda el modelo y recuerda hasta qué día ya entrenó, para no reentrenar de cero cada vez que se corra el notebook con los mismos datos.

In [ ]:
import json
from pathlib import Path

RUTA_MODELO_TRAMO = Path("modelo_tramo.txt")
RUTA_MARCA_TRAMO = Path("modelo_tramo_marca.json")


def _leer_marca():
    if RUTA_MARCA_TRAMO.exists():
        with open(RUTA_MARCA_TRAMO, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


marca_previa = _leer_marca()
if marca_previa is not None and dias[-1] <= pd.Timestamp(marca_previa["hasta"]).date():
    print(f"Sin dias mas nuevos que la ultima corrida ({marca_previa['hasta']}) -> NO se reentrena.")
    modelo_tramo = lgb.Booster(model_file=str(RUTA_MODELO_TRAMO))
    USA_RATIO = marca_previa.get("usa_ratio", True)
else:
    modelo_tramo.save_model(str(RUTA_MODELO_TRAMO))
    with open(RUTA_MARCA_TRAMO, "w", encoding="utf-8") as f:
        json.dump({"hasta": str(dias[-1]), "usa_ratio": USA_RATIO}, f)
    print(f"Modelo de tramos guardado, con datos hasta {dias[-1]} "
          f"(objetivo: {'ratio' if USA_RATIO else 'conteo directo'}).")

### Mapas de comparación: real vs. predicho (días de test)

Para ver si el modelo se parece a la realidad: se suma, tramo por tramo, cuántos camiones distintos pasaron **realmente** y cuántos **predijo** el modelo, en esos mismos días de test (nunca usados para entrenar). Mismo estilo de mapa de calor de rutas: color por cantidad de camiones distintos, rojo = pocos, verde = muchos; selector de capa (calles/satélite) y leyenda con el degradé.

In [ ]:
import folium
import matplotlib
import matplotlib.colors as mcolors
from branca.element import MacroElement, Template

test_pred = test.copy()
test_pred["conteo_predicho"] = predecir_conteo(test_pred)

agregado_test = (test_pred.groupby("clave")
                  .agg(real=("conteo", "sum"), predicho=("conteo_predicho", "sum"))
                  .reset_index())
agregado_test = agregado_test[(agregado_test["real"] > 0) | (agregado_test["predicho"] > 0)]

print(f"Tramos comparados (dias de test): {len(agregado_test):,}")
print(f"Total camiones-dia REAL: {agregado_test['real'].sum():.0f}  |  "
      f"Total camiones-dia PREDICHO: {agregado_test['predicho'].sum():.0f}")

# Para DIBUJAR no se usa el promedio por tramo (eso colapsa cruces/zonas con muchas
# direcciones distintas en un punto): se usa la geometria real de cada segmento GPS de los
# dias de test, coloreada segun el valor agregado (real o predicho) de su tramo.
dias_test = sorted(test["dia"].unique())
segmentos_test = validos.loc[
    validos["dia"].isin(dias_test),
    ["clave", "latitude", "longitude", "lat_sig", "lon_sig"],
].rename(columns={"latitude": "lat1", "longitude": "lon1", "lat_sig": "lat2", "lon_sig": "lon2"})
segmentos_test = segmentos_test.merge(agregado_test, on="clave", how="inner")

print(f"Segmentos dibujados (geometria real, dias de test): {len(segmentos_test):,}")


def construir_mapa_tramos(tabla_mapa, columna_valor, nombre_archivo, etiqueta, titulo):
    """Mapa de calor de rutas: cada segmento real coloreado segun `columna_valor` de su tramo
    (rojo = pocos camiones distintos, verde = muchos)."""
    cmap = matplotlib.colormaps["RdYlGn"]
    max_valor = max(tabla_mapa[columna_valor].max(), 1)
    norm = mcolors.Normalize(vmin=0, vmax=max_valor)

    mp = folium.Map(location=[mov["latitude"].mean(), mov["longitude"].mean()], zoom_start=6,
                     tiles=None, prefer_canvas=True)
    folium.TileLayer("OpenStreetMap", name="Calles y lugares (OpenStreetMap)", control=True).add_to(mp)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        name="Satélite (Esri)", control=True,
    ).add_to(mp)

    capa = folium.FeatureGroup(name="Uso de rutas", show=True)
    for _, t in tabla_mapa.sort_values(columna_valor).iterrows():
        color = mcolors.to_hex(cmap(norm(t[columna_valor])))
        folium.PolyLine(
            [(t["lat1"], t["lon1"]), (t["lat2"], t["lon2"])],
            color=color, weight=4, opacity=0.85,
            tooltip=f"{t[columna_valor]:.1f} camiones distintos ({etiqueta})",
        ).add_to(capa)
    capa.add_to(mp)
    folium.LayerControl(collapsed=False).add_to(mp)

    titulo_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9999;
            background: white; padding: 8px 20px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.4); font-size: 16px; font-weight: bold;
            white-space: nowrap;">
    __TITULO__
</div>
{% endmacro %}
""".replace("__TITULO__", titulo)
    titulo_elem = MacroElement()
    titulo_elem._template = Template(titulo_html)
    mp.get_root().add_child(titulo_elem)

    leyenda_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; bottom: 30px; right: 10px; z-index: 9999;
            background: white; padding: 10px 14px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.4); font-size: 13px; width: 260px;">
    <div style="margin-bottom: 6px;">__ETIQUETA__ — camiones distintos (días de test, rojo = pocos, verde = muchos)</div>
    <div style="height: 12px; border-radius: 3px;
                background: linear-gradient(to right, #a50026, #f98e52, #ffffbf, #86cb66, #1a9850);"></div>
    <div style="display:flex; justify-content: space-between;">
        <span>0</span><span>__MAX__</span>
    </div>
</div>
{% endmacro %}
""".replace("__MAX__", f"{max_valor:.1f}").replace("__ETIQUETA__", etiqueta)

    leyenda = MacroElement()
    leyenda._template = Template(leyenda_html)
    mp.get_root().add_child(leyenda)

    mp.save(nombre_archivo)
    return mp


# Mapa 1: datos REALES (lo que efectivamente paso en los dias de test)
mp_real = construir_mapa_tramos(segmentos_test, "real", "mapa_calor_rutas_real.html", "REAL",
                                 titulo="Este es el mapa con datos reales")
mp_real

In [ ]:
# Mapa 2: datos PREDICHOS por el modelo (lo que aprendio el machine learning) para esos mismos dias de test
mp_predicho = construir_mapa_tramos(segmentos_test, "predicho", "mapa_calor_rutas_prediccion.html", "PREDICHO",
                                     titulo="Este es el mapa de predicción")
mp_predicho